**Tabla de contenido**

- [Introducción](#Introduccio)
- [Librerías](#Librerias)
- [Preprocesamiento](#Preprocesamiento)
- [Preparación de los datos para modelado](#Preparacion-de-los-datos-para-modelado)
- [Modelado](#Modelado)

# Introduccion

Tu tarea consiste en crear un clasificador binario que prediga si un comentario de Reddit infringe una norma específica. El conjunto de datos procede de una gran colección de comentarios moderados, con una serie de normas de subreddit, tonos y expectativas de la comunidad.

`dataset`
- **body** - el texto del comentario
- **rule** - la regla que se considera que infringe el comentario
- **subreddit** - el foro en el que se hizo el comentario
- **positive_example_{1,2}** - ejemplos de comentarios que infringen la regla
- **negative_example_{1,2}** - ejemplos de comentarios que no infringen la regla
- **rule_violation** - el objetivo binario


# Librerias

In [ ]:
import os
import pandas as pd
import re

In [ ]:
file_path = lambda file: os.path.join(os.getcwd(),'data/Agile Community Rules Classification',file)
train = pd.read_csv(file_path('train.csv'))
#train = train.set_index('row_id', drop=True)
#pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
train.head(2)

In [ ]:
print(train['positive_example_1'][4])

# Preprocesamiento

Vamos a prepar los datos para el modelo Bertweet. BERTweet es un modelo de lenguaje basado en la arquitectura BERT (Bidirectional Encoder Representations from Transformers), pero específicamente entrenado en tweets (textos de Twitter) en ingles. Está optimizado para lenguaje informal, lo que incluye jerga de redes sociales, hashtags, emoticonos, menciones (@) y ortografía no estándar (ej: "loooove"). Tiene un Tokenizador adaptado: Maneja mejor palabras repetidas ("goooool"), contracciones ("don't" → "do n't") y palabras concatenadas ("NewYork").

Este modelo está diseñado para tareas de Procesamiento de Lenguaje Natural (NLP) en redes sociales, como:

1. `Clasificación de Texto`

- Análisis de sentimiento (ej.: ¿Es un tweet positivo, negativo o neutro?).
- Detección de hate speech, spam o bullying.
- Identificación de noticias falsas (fake news) en redes sociales.

2. `Extracción de Información`

- Named Entity Recognition (NER): Identificar personas, lugares, etc., en tweets.
- Detección de temas (topic modeling) en conversaciones de Twitter.

3. `Aplicaciones Específicas`

- Moderación automática de contenido en plataformas sociales.
- Respuesta a preguntas (QA) en contextos informales.
- Generación de texto (aunque no es su enfoque principal).

Esto implica que el preprocesamiento de los textos debe realizarse de la siguiente forma:

1. Reemplazar las URL por la abreviatura `[URL]`.
2. Reemplazar las mensiones de usarios por la abreviatura `[USER]`
3. Los `Hashtag` deben dejarse tal cual como están.
4. Los emojis deben dejarse ya que ayudan al modelo a entender tono emosional o sarcasmo.
5. Se deben eliminar múltiples espacios, tabs o saltos de linea innecesarios.
6. No convertir a mayúscula o minúscula, ni eliminar los signos de puntuación. Este modelo no distigue entre mayúscula/minúscula.

In [ ]:
from urlextract import URLExtract

def cleantext_toBERTweet(text):
    text = re.sub(r"\s+"," ", text).strip()                     # reemplaza múltiples espacios, tabs o saltos de línea por un solo espacio
    text = re.sub(r"@\w+", " @USER ", text)
    email_pattern = r'\b([A-Za-z0-9._%+-]+)\s*(?:@|\[at\]|\(at\)|arroba)\s*([A-Za-z0-9.-]+)\s*(?:\.|\[dot\]|\(dot\)|punto)\s*([A-Za-z]{2,})\b'
    text = re.sub(email_pattern,'@EMAIL',text)                 # reemplaza correo electrónicos a [EMAIL]
    phone_pattern = r'(?<!\w)(?:\+?\d{1,3}|\(\+?\d{1,3}\))?(?:[-. /]?\d{2,4}){2,5}(?:[-. /]?\d{2,})\b(?:[ ]*(?:ext|xtn|x|#)[ ]*\d{1,6})?(?!\w)'
    text = re.sub(phone_pattern, '@PHONE', text)               # Reemplaza números de teléfonos por [PHONE]

    # Reemplazo de URLs estándar detectadas por URLExtract
    extractor = URLExtract()
    urls = extractor.find_urls(text)
    for url in urls:
        text = text.replace(url, 'HTTPURL')  

    # Patrón “raro” en url (/p/... .xxx)
    pattern_url_raro =  r"/[A-Za-z]/[\w-]+\.[A-Za-z]{2,4}\b"
    text = re.sub(pattern_url_raro, "HTTPURL", text)
    # Cualquier HTTP/HTTPS
    pattern_any_url = r"(?:https?://|://)[^\s]+"
    text = re.sub(pattern_any_url, "HTTPURL", text)
    # patrones raros
    pattern_scheme = r"\b[a-z][\w+.-]*://[^\s]+\b"
    text = re.sub(pattern_scheme, "HTTPURL", text)

    # “come” todas las aperturas de paréntesis o corchetes adyacentes antes de un token del tipo […]
    pattern =  r'([(\[])\[HTTPURL\]([)\]])'  # Captura ( [URL] ) o [ [URL] ]
    text = re.sub(pattern,'HTTPURL',text)
    text = re.sub(r"\*", "", text)
    text = re.sub(r'\/','',text)
    return text

In [ ]:
#train = train.set_index('row_id', drop=True)
pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
muestra = train['negative_example_2'].sample(n=10)
muestra.head(10)

In [ ]:
muestra = muestra.apply(cleantext_toBERTweet)
muestra.head(10)

Perfecto, ahora tenemos que enfocarnos en los emojis.

In [ ]:
from emoji import is_emoji
from nltk.tokenize import TweetTokenizer

tokenizer = TweetTokenizer()

# Diccionario de emoticonos ASCII comunes
ascii_emoticons = {
    ":)", ":-)", ":(", ":-(", ":/", ":-/", ":D", ":-D",
    ";)", ";-)", ":P", ":-P"
}

def normalizeToken(token):
    # Eliminamos emoticonos ASCII
    if token in ascii_emoticons:
        return ""
    
    # Eliminamos emojis Unicode
    elif any(is_emoji(char) for char in token):
        return ""
    
    # Normalizamos algunos símbolos tipográficos
    elif token == "’":
        return "'"
    elif token == "…":
        return "..."
    else:
        return token

def normalize_repetitions(text):
    # Reemplazar secuencias largas de "!" por máximo "!!!"
    text = re.sub(r"!{4,}", "!", text)
    
    # Reemplazar secuencias largas de "." por máximo "..."
    text = re.sub(r"\.{4,}", ".", text)
    
    # Normalizar secuencias de ">" (con o sin espacios) a un solo ""
    text = re.sub(r"(>\s*){2,}", " ", text)

    # Para cualquier otro símbolo repetido (2 o más), dejar solo 1
    text = re.sub(r"([^a-zA-Z0-9\s])\1{1,}", r"\1", text)
    text = re.sub(r'[|=]', ', ', text)
    
    return text

def normalize_punctuation_spacing(text):
    # Quitar espacio antes de puntuación
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Asegurar espacio después de puntuación, excepto al final
    text = re.sub(r"([,.!?;:])(?!\s|$)", r"\1 ", text)

    # Normalizar espacios múltiples otra vez
    text = re.sub(r"\s+", " ", text).strip()

    return text

def cleantext_toBERTweetEmoji(text):
    # Normalización básica de espacios
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenización estilo tweet
    tokens = tokenizer.tokenize(text.replace("’", "'").replace("…", "..."))
    
    # Normalización de tokens (sin emojis ni emoticonos)
    normTweet = " ".join(
        [normalizeToken(token) for token in tokens if normalizeToken(token) != ""]
    )

    # Normalización de repeticiones (incluye ">" y demás símbolos)
    normTweet = normalize_repetitions(normTweet)

    # Normalización de puntuación
    normTweet = normalize_punctuation_spacing(normTweet)

    # Reglas de contracciones (basadas en BERTweet)
    normTweet = (
        normTweet.replace("cannot ", "can not ")
        .replace("n't ", " n't ")
        .replace("n 't ", " n't ")
        .replace("ca n't", "can't")
        .replace("ai n't", "ain't")
        .replace("You ' re", "You're")
    )
    normTweet = (
        normTweet.replace("'m ", " 'm ")
        .replace("'re ", " 're ")
        .replace("'s ", " 's ")
        .replace("'ll ", " 'll ")
        .replace("'d ", " 'd ")
        .replace("'ve ", " 've ")
    )
    normTweet = (
        normTweet.replace(" p . m .", " p.m.")
        .replace(" p . m ", " p.m ")
        .replace(" a . m .", " a.m.")
        .replace(" a . m ", " a.m ")
    )

    return " ".join(normTweet.split())

In [ ]:
muestra = muestra.apply(cleantext_toBERTweetEmoji)
muestra.head(10)

Veamos ahora como quedan los texto, para esto sacaremos muestras aleatorias y las limpiaremos, esto con el fin de saber que todo está ok.

Podemos ver que la función parece funcionar correctamente. Apliquemosla al set de datos.

In [ ]:
df_train_ob = train.select_dtypes(['object'])
for col in df_train_ob.columns:
    df_train_ob[col]=df_train_ob[col].apply(cleantext_toBERTweet)
df_train_ob.head()

Perfecto, ahora nos enfocamos en tratar los emojis para que el modelo los entienda.

In [ ]:
df_train_ob = df_train_ob.select_dtypes(['object'])
for col in df_train_ob.columns:
    df_train_ob[col]=df_train_ob[col].apply(cleantext_toBERTweetEmoji)
df_train_ob.head()

# Preparacion de los datos para modelado

Dado que nuestro objetivo es entrenar el modelo BERTweet para que logre identificar si un comentario viola las reglas, debemos hacer lo siguiente:

`Tokens especiales: BERTweet usa los mismos tokens especiales que BERT:`

- [CLS] al inicio.

- [SEP] para separar segmentos.


In [ ]:
df_train_ob['combined_text'] = (
    "[CLS] comment: " + df_train_ob['body'] + 
    " [SEP] rule: " + df_train_ob['rule'] + 
    " [SEP] positive_examples1: " + df_train_ob['positive_example_1'] +
    " [SEP] positive_examples2: " + df_train_ob['positive_example_2'] +
    " [SEP] negative_examples1 " + df_train_ob['negative_example_1'] +
    " [SEP] negative_examples2 " + df_train_ob['negative_example_2'] +
    " [SEP]"
)
df_train_ob['combined_text'][0]


Perfecto, Ahora necesitamos tokenizar el texto y consultar cual es el token con mayor longitud

In [ ]:
from transformers import AutoTokenizer  # Mejor usar AutoTokenizer para mayor flexibilidad

# Cargar tokenizador (usa el correcto para tu modelo)
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base")  # Ejemplo con Bertweet

# Función para contar tokens con manejo de errores
def count_tokens(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(tokenizer.tokenize(text))

# Aplicar a cada fila
df_train_ob['token_length'] = df_train_ob['combined_text'].apply(count_tokens)

In [ ]:
df_train_ob.head(1)

In [ ]:
max_token_row = df_train_ob.loc[df_train_ob['token_length'].idxmax()]

print("Texto con más tokens:")
print(max_token_row['combined_text'])
print(f"\nNúmero de tokens: {max_token_row['token_length']}")

Perfecto, hay un texto que contiene 473 tokens. Esto supera el token máximo permitido por Bertweet, que es de 128. Esto indica que debo usar berttweet large.

# Modelado

Antes de entar a modelar, es necesario saber si las clases estan balanceadas.

In [ ]:
train['rule_violation'].value_counts(normalize=True)*100

En teoria las clases están balanceadas.

`Los modelos de transformadores como DistilBERT no pueden recibir cadenas de texto sin procesar como entrada; en su lugar, asumen que el texto ha sido tokenizado y codificado como vectores numéricos`. `La tokenización es el paso de descomponer una cadena en las unidades  utilizadas en el modelo`. Pero antes de tokenizar creemos el dataframe que contiene lo que necesitamos.


In [ ]:
df_train = pd.concat([df_train_ob['combined_text'],train['rule_violation']],axis=1)
df_train.head(1)

Perfecto, ahora necesitamos llevar estos datos a un formato adecuado.

Este código prepara los textos y sus etiquetas para que un modelo BERTweet pueda entrenarse en una tarea de clasificación (detección de violaciones de reglas, probablemente en tweets). Convierte texto a tensores, los organiza en un dataset compatible con PyTorch y deja todo listo para usar con Hugging Face Trainer

In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import torch, gc
# Tokenización
# 1. Cargar tokenizer (usando la versión fast si es compatible)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)  # use_fast=True es más eficiente

# 2. Preprocesamiento y tokenización más eficiente
def tokenize_data(texts, max_length=360):
    return tokenizer(
        texts,
        truncation=True,
        padding='longest',  # Padding dinámico hasta la máxima longitud en el batch
        max_length=max_length,
        return_tensors="pt"  # Devuelve tensores PyTorch directamente
    )

# 3. Dataset optimizado
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        # Ya tenemos tensores, no necesitamos conversión
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
        
    def __len__(self):
        return len(self.labels)

# 4. Procesamiento más eficiente
# Primero dividimos (es más eficiente tokenizar después)
X_train, X_val, y_train, y_val = train_test_split(
    df_train['combined_text'].tolist(),
    df_train['rule_violation'].tolist(),
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# Tokenización con batches para mejor rendimiento
train_encodings = tokenize_data(X_train)
val_encodings = tokenize_data(X_val)

# Crear datasets
train_dataset = CustomDataset(train_encodings, y_train)
val_dataset = CustomDataset(val_encodings, y_val)


Perfecto, ahora lo que tenemos que hacer es 

In [ ]:
from transformers import AutoModelForSequenceClassification
model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_labels = 2
model = (AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels).to(device))

Para monitorear las métricas durante el entrenamiento, necesitamos definir una función compute_metrics() para el Entrenador. Esta función recibe un objeto EvalPrediction (que es una tupla con nombre con atributos de predicciones y label_ids) y debe devolver un diccionario que mapea el nombre de cada métrica a su valor. Para nuestra aplicación, calcularemos la puntuación F1 y la precisión del modelo de la siguiente manera:

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

# Ajustes de dropout en DistilBERT
model.config.hidden_dropout_prob = 0.1
if hasattr(model.config, "attention_dropout_prob"):
    model.config.attention_dropout_prob = 0.1


batch_size = 15  # Redujimos de 8 a 4 (o incluso 2 si persiste el error)
model_name = f"{model_ckpt}-finetuned-violacion"

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir=model_name,
    num_train_epochs=10,  # Menos épocas
    learning_rate=2e-5,  
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.0,  
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    gradient_accumulation_steps=2,  # Opcional si el batch es muy pequeño
    disable_tqdm=False,
    push_to_hub=False,
    log_level="error",
    save_total_limit=2,
)


In [ ]:
from transformers import DataCollatorWithPadding

# Asegúrate de que el tokenizer tiene una longitud máxima manejable
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # Early Stopping
)


In [ ]:
#del X_train, X_val
gc.collect(); torch.cuda.empty_cache()
trainer.train()

In [ ]:
from transformers import AutoModelForSequenceClassification

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        model_ckpt,
        num_labels=2   # ajusta según tu problema (binario, multi-clase, etc.)
    )

trainer = Trainer(
    model_init=model_init,   # OJO aquí
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
import optuna
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "num_train_epochs": trial.suggest_categorical("num_train_epochs", [8]),
        #"num_train_epochs": trial.suggest_int("num_train_epochs", 2, 12), 
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
    }

best_run = trainer.hyperparameter_search(
    direction="maximize",     # queremos maximizar accuracy
    backend="optuna",
    n_trials=10,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_accuracy"],  # criterio de selección
)

In [ ]:
print("Mejor combinación encontrada:")
print(best_run)